# 项目启动与数据接入
运营商给了我们 沈阳 5,644 个手机用户在 2018 年 10 月 3 日（国庆第三天）一整天的 53 万条信令（已脱敏）。 我们需要做完一个完整的数据项目：先把数据验收清楚，再识别停留和出行、定义指标口径、得出结论， 最后把这套分析改造成「每天能自动运行、坏数据进不来、每个数字能追溯、能对外展示」的系统。

## 手上有什么数据

全部装在 `data/signal.db` 里（SQLite 单文件库）。**学生拿到的就是这个库，没有原始 CSV**——企业里分析师面对的是数仓里的表，原始文件留在数据平台，这和真实工作一致。
`ods_` = 原始层，**一个值都没改**，入库时只做了起列名、补顶点序号这类不改变含义的事。

| 表 | 行数 | 是什么 |
|---|---|---|
| `ods_signal` | 532,828 | 信令：谁（IMSI）在什么时刻连了哪个基站扇区 |
| `ods_cell` | 49,954 | 基站扇区的经纬度——信令里只有编号，位置全靠它补 |
| `ods_poi_purpose` | 1,273 | 兴趣点：居住、商业、办公…… |
| `ods_poi_transport` | 968 | 地铁站、公交站 |
| `ods_aoi_v1 / v2 / json` | 104 / 1,685 / 1,716 | 商圈边界的**三个版本**，D3 要甄别该信哪个 |
| `ods_district_boundary` | 2,951 | 沈阳各区县的边界顶点 |
| `ods_nid_mapping` | 24 | `nid` 字段的字典（D2 选讲里要验证它可不可信） |

### 数据在项目当中如何流动

![数据流动](image/数据流动.png)

### 项目需求
- 客户需求：国庆期间想要了解一下区里的夜间经济情况，运行商有数据，你帮我们看看

一个能回答的问题至少要能说清四样东西：看那个对象，看什么指标，看哪段时间，跟谁比

### 环境准备
第一步：先保证环境是OK的

In [ ]:
import sqlite3  # 读取sqlite数据库的
from pathlib import Path # 现代化处理路径的

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from signal_insight import config
from signal_insight import viz

DB_PATH = config.DB_PATH
IMG_DIR = config.IMAGE_DIR
IMG_DIR.mkdir(exist_ok = True)

assert DB_PATH.exists(), f"找不到数据库文件{DB_PATH}"

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
con = sqlite3.connect(DB_PATH)    # 整个 notebook 共用一个数据库连接

In [ ]:
# 自检：版本、中文字体、数据库大小
import sys, matplotlib, seaborn
from matplotlib import font_manager
viz.apply_theme("business")
installed = {f.name for f in font_manager.fontManager.ttflist}
cn_font = list(f for f in plt.rcParams["font.family"] 
            if f in installed and f != "DejaVu Sans")

print(f"Python      {sys.version.split()[0]}")
print(f"pandas      {pd.__version__}")
print(f"numpy       {np.__version__}")
print(f"matplotlib  {matplotlib.__version__}")
print(f"seaborn     {seaborn.__version__}")
print(f"SQLite      {sqlite3.sqlite_version}")
print(f"中文字体    {cn_font}")
print(f"数据库      {DB_PATH.name}（{DB_PATH.stat().st_size / 1e6:.0f} MB）")

## 数据接入
### 连上数据库加载原始数据到pandas

In [ ]:
# 连接数据库读取数据到pandas

tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name", con)
tables['行数'] = [
    pd.read_sql(f"SELECT COUNT(*) AS n FROM {t}", con)["n"][0]
    for t in tables["name"]
]

raw = pd.read_sql("SELECT * FROM ods_signal", con)
raw.head()

In [ ]:
assert raw.shape == (532_828, 10), f"行列数不对： {raw.shape},应该是：(532828, 10)"
print("数量核对准确： 532828行 x 10列")

### 数据画像：每个字段张什么样子

需求：写一个函数profile(df),每个字段输出一行：类型，缺失数，缺失率，唯一值的个数，一个示例值.最终返回一个dataframe

In [ ]:
def profile(df: pd.DataFrame) -> pd.DataFrame:
    """生成数据画像，每个字段输出一行统计信息。

    缺失值包括 NaN/None/NA，以及文本类字段（str、object、category）中
    去除首尾空白后为空的字符串。唯一值个数和示例值均基于非缺失值计算，
    保证各统计项的缺失口径一致。

    Args:
        df: 待分析的 DataFrame。允许 0 行、重复列名，以及包含 list、dict
            等不可哈希值的列。

    Returns:
        每个字段一行的 DataFrame，包含以下列：

        - 字段：列名。
        - 类型：pandas dtype 的字符串表示，如 "int64"、"str"。
        - 缺失数：缺失值个数（int）。
        - 缺失率：缺失数 / 总行数（float），0 行时为 0.0。
        - 唯一值个数：非缺失值的去重个数（int）。不可哈希的值按其字符串
          形式去重。
        - 示例值：第一个非缺失值；全部缺失时为 None。

    Examples:
        >>> df = pd.DataFrame({"a": [1, None, 3], "b": ["x", " ", None]})
        >>> profile(df)
          字段       类型  缺失数       缺失率  唯一值个数 示例值
        0  a  float64    1  0.333333      2  1.0
        1  b      str    2  0.666667      1    x
    """
    n = len(df)
    rows = []
    for col, s in df.items():  # 按位置取列，列名重复也不会出错
        missing = s.isna()
        if (pd.api.types.is_string_dtype(s) or s.dtype == object
                or isinstance(s.dtype, pd.CategoricalDtype)):
            blank = s.astype("string").str.strip().eq("").fillna(False).astype(bool)
            missing = missing | blank
        valid = s[~missing]
        try:
            n_unique = int(valid.nunique())
        except TypeError:  # 列里有 list/dict 等不可哈希的值
            n_unique = int(valid.astype(str).nunique())
        n_missing = int(missing.sum())
        rows.append({
            "字段": col,
            "类型": str(s.dtype),
            "缺失数": n_missing,
            "缺失率": n_missing / n if n else 0.0,
            "唯一值个数": n_unique,
            "示例值": valid.iloc[0] if len(valid) else None,
        })
    return pd.DataFrame(rows)

In [ ]:
prof = profile(raw)
# 对账：两列全空，imsi 缺 15 个
p = prof.set_index("字段")
assert p.loc["tmp0", "缺失数"] == 532_828 and p.loc["tmp1", "缺失数"] == 532_828
assert p.loc["imsi", "缺失数"] == 15
print("数据核对通过：tmp0、tmp1 全部为空；imsi 缺 15 个")

In [ ]:
prof

![数据字段解释](image/数据字段解释.png)

### 时间戳转化为北京时间

需求：timesmap是13位证书，把它转化为北京时间，看下数据的跨度，从哪天到哪天

In [ ]:
# 时间的转换： 13位整数一般是从1970-01-01 00:00 UTC起的毫秒数
raw['ts'] = pd.to_datetime(raw['timestamp'], unit='ms') + pd.Timedelta(hours=8)
raw[['timestamp', 'ts']].head(3)

In [ ]:
raw['ts'].agg(['min','max'])

In [ ]:
# 对账：时间范围
assert str(raw["ts"].min())[:16] == "2018-10-01 12:52" and str(raw["ts"].max())[:16] == "2018-10-03 23:57"
print("数据核对通过：2018-10-01 12:52 至 2018-10-03 23:57")

### 每天有多少数据
需求： 时间横跨三天，每天都数据量均匀吗？

固定做法：先算中间表，核对无误后写回库里，画图时从库里去，中间表能看，能核对，能留档，画图代码只负责画，不负责算

In [ ]:
daily = (raw.assign(日期 = raw['ts'].dt.strftime('%m-%d'))
    .groupby('日期')
    .agg(记录数=('timestamp', "size"), 用户数=('imsi', 'nunique'))
).reset_index()

daily['记录占比'] = daily['记录数'] / daily['记录数'].sum()
daily.to_sql("ads_daily_volume", con, if_exists="replace", index=False)

::: NOTE ADS应用层
- 是什么：ADS 是数据仓库分层里的应用数据层，直接服务某个应用，比如平台报告接口的结果表、BI 看板结果表、对外 API 查询表。它通常已经经过 DWD/DWS 加工，聚合到非个人粒度，例如按天、地区、渠道、门店、商品、指标汇总，可以对外展示。
- 为什么需要它：ADS 把数据聚合到非个人粒度，可以做好隐私抑制，比如小样本屏蔽、阈值控制、泛化、脱敏，避免泄露个人信息。它通常是只读结果表，查询很快，因为已经预计算、预聚合，不要求用户现算复杂逻辑。同时还能统一指标口径，避免每个应用各算一套。
- 企业里面：ADS 常常落到 OLAP 引擎，比如 ClickHouse、Doris、StarRocks、ADB，或者直接被 BI 工具读取。需要考虑查询速度，比如分区、排序键、物化视图、预聚合、宽表设计；也需要考虑权限，比如 RBAC、行级权限、列级权限、脱敏、审计、数据范围控制。
:::

::: IMPORTANT OLTP和OLAP
- 是什么：OLTP 和 OLAP 是什么？它们是两种数据库负载。OLTP 是 Online Transaction Processing，在线事务处理，特点是很多很小的请求，比如下单、扣费、支付、修改用户信息。OLAP 是 Online Analytical Processing，在线分析处理，特点是查询数量少，但单次查询很大，比如全市每小时多少人、过去一年各渠道销售额、用户行为漏斗分析。
- 为什么要分这两种：两种负载对数据库的要求相反。OLTP 要求单条写入快、不出错、强一致、高并发、低延迟，通常按行存储，用 B+ 树索引，适合点查和短事务。OLAP 要求整列扫描快、聚合快、压缩率高、吞吐大，通常按列存储，用向量化执行、MPP、预聚合、物化视图，适合大宽表分析。所以企业里面业务库和分析库是分开的，后者常用列式存储，通过 CDC、ETL、ELT 把数据同步过去。
- 面试时：MySQL 和 ClickHouse 有什么区别？怎么答？答题骨架实际上就是 OLTP 和 OLAP 的区别。可以这样回答：MySQL 是典型 OLTP 数据库，行存，B+ 树索引，支持事务、外键、强一致，适合订单、交易、用户等高频点查点写业务。ClickHouse 是典型 OLAP 数据库，列存，MergeTree 引擎，向量化执行，压缩比高，适合海量数据的大宽表扫描和聚合分析，比如日志、行为、报表。业务交易走 MySQL，分析报表走 ClickHouse，二者通过 CDC/ETL 同步。如果追问“ClickHouse 能不能更新”，答：可以支持轻量更新或异步变更，但不适合高频单行更新和事务场景；如果追问“为什么 ClickHouse 聚合快”，答：列存只读需要的列、压缩减少 IO、向量化执行、并行扫描、预聚合和物化视图。最后总结：不是谁替代谁，而是 OLTP 和 OLAP 分工不同。
:::

拿到中间表之后，很自然的想法就是通过可视化的形式展示下每天都数量相关内容

In [ ]:
d = pd.read_sql("SELECT * FROM ads_daily_volume", con)

In [ ]:
# pip install tabulate
# print(d.to_markdown())

In [ ]:
import math

def fmt_pct(p: float) -> str:
    """把比例格式化为百分比字符串，遵循图表规范第 6 节。

    不小于 10% 时取整；1%–10% 保留一位小数；小于 1% 时保留到第一位
    有效数字，避免出现 "0.0%"。

    Args:
        p: 比例，取值 0–1。

    Returns:
        百分比字符串，如 "97%"、"2.7%"、"0.01%"。
    """
    pct = p * 100
    if pct == 0:
        return "0%"
    if pct >= 10:
        return f"{pct:.0f}%"
    if pct >= 1:
        return f"{pct:.1f}%"
    return f"{pct:.{-math.floor(math.log10(pct))}f}%"


def fmt_val(col: str, v: float, share: float) -> str:
    """生成条形末端的数值标注。

    Args:
        col: 指标列名，决定数值格式。
        v: 要标注的数值。
        share: 当日记录占比，仅在 col 为 "记录数" 时使用。

    Returns:
        标注文字，整数带千分位，记录数附带占比。
    """
    if col == "记录数":
        return f"{v:,.0f}（{fmt_pct(share)}）"
    if col == "用户数":
        return f"{v:,.0f}"
    return f"{v:.0f}" if v >= 10 else f"{v:.1f}"


# ---- 元信息：按实际情况修改 ----
DAY, FIG_NO = 1, 1
SCRIPT = "D01_数据概览"
AUTHOR = "Azurekite 2769433"

# ---- 计算：标题里的每个数字都来自这里 ----
d = d.sort_values("日期").reset_index(drop=True)
d["人均记录"] = d["记录数"] / d["用户数"]

top = d.loc[d["记录数"].idxmax()]                 # 记录最多的一天
rest = d.drop(index=top.name)
ref = rest.loc[rest["人均记录"].idxmax()]         # 其余日期中人均最高的一天
low = d.loc[d["记录数"].idxmin()]
ratio = top["人均记录"] / ref["人均记录"]

title = (f"{top['日期']} 记录占 {fmt_pct(top['记录占比'])}，人均 {top['人均记录']:.0f} 条，"
         f"是 {ref['日期']} 的 {ratio:.0f} 倍")
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = (f"沈阳移动信令样本 {d['日期'].iloc[0]} 至 {d['日期'].iloc[-1]}，按日期统计全部记录；"
            f"用户按日去重，各日分别计数、不能相加\n"
            f"人均记录 = 记录数 ÷ 用户数；斜线 = 10月2日数据不完整、未去重；"
            f"{low['日期']} 仅 {low['记录数']:,} 条，"
            f"各日数据量差异可能来自样本抽取范围，不代表客流变化")
source = (f"数据来源：沈阳移动信令样本（脱敏，2018 年国庆）；"
          f"脚本：{SCRIPT}；制图:{AUTHOR}")

# ---- 画图：三个并列面板共用日期轴，代替双 Y 轴 ----
W, H = 9.6, 4.6
fig, axes = plt.subplots(1, 3, figsize=(W, H), sharey=True, dpi=100)
fig.subplots_adjust(left=0.08, right=0.97, top=1 - 1.6 / H,
                    bottom=0.55 / H, wspace=0.12)

panels = [("记录数", "记录数（条）"), ("用户数", "用户数（人）"),
          ("人均记录", "人均记录（条/人）")]
is_top = d["日期"].eq(top["日期"]).to_numpy()
colors = np.where(is_top, viz.MAIN, viz.GRAY)              # 灰色打底，主色强调一天
y = np.arange(len(d))

# 10 月 2 日：数据不完整、未去重，用白色斜线标记（沿用 D1_01 的视觉约定）
# 按实际日期格式调整匹配：字符串 "2018-10-02" 用 endswith；"10月2日" 用 contains
is_hatch = d["日期"].astype(str).str.endswith("10-02").to_numpy()

for ax, (col, label) in zip(axes, panels):
    vals = d[col].to_numpy(dtype=float)
    vmax = vals.max()
    ax.barh(y, vals, color=colors, height=0.6)
    if is_hatch.any():                             # 只给 10/2 叠一层白色斜线
        ax.barh(y[is_hatch], vals[is_hatch], color=colors[is_hatch],
                height=0.6, hatch="////", edgecolor="white", linewidth=0.5)
    ax.set_xlim(0, vmax * 1.05)                    # 条形从 0 开始
    for yi, v, hl, share in zip(y, vals, is_top, d["记录占比"]):
        text = fmt_val(col, v, share)
        if v > 0.55 * vmax:                        # 长条：数字写在条内
            ax.text(v - vmax * 0.02, yi, text, ha="right", va="center",
                    fontsize=10, color="white" if hl else viz.INK,
                    fontweight="bold" if hl else "normal")
        else:                                      # 短条：数字写在条外
            ax.text(v + vmax * 0.02, yi, text, ha="left", va="center",
                    fontsize=10, color=viz.MUTED)
    ax.set_title(label, loc="left", fontsize=10.5, color=viz.INK, pad=6)
    ax.grid(False)
    ax.xaxis.set_visible(False)                    # 数值已直接标注
    ax.spines["bottom"].set_visible(False)
    ax.axvline(0, color=viz.FAINT, lw=0.8)

axes[0].set_yticks(y, d["日期"])
axes[0].invert_yaxis()                             # 日期从上到下按时间顺序

# ---- 收尾：强调色短条、标题、副标题、来源行（D01 手写，D02 起由 viz.finish 完成）----
fname = f"D{DAY:02d}_{FIG_NO:02d}_各日记录分布.png"
viz.finish(fig,
        title=title,
        subtitle=subtitle,
        source=source,
        path=fname
          )

plt.show()

报告里可以这样写，数据时间范围为2018年10月1日至3日。但97.3% (518,443)条的记录集中在10月3日; 10月1日仅54条，10月2日约有1.4万条且为零散记录，这个可以具体看图D1_01,因此本报告的分析范围将会限定在10月3日这一天

In [ ]:
# ---- 读回断言：从图对象里把数读回来，和中间表核对——靠代码，不靠眼睛 ----
for ax, (col, _) in zip(axes, panels):
    widths = [b.get_width() for b in ax.containers[0]]   # containers[0] = 主条形那一层
    assert np.allclose(widths, d[col].to_numpy(dtype=float)), \
        f"{col} 条形长度和中间表对不上"

assert fmt_pct(top["记录占比"]) in fig.texts[0].get_text(), \
    "标题里的占比不是从数据算出来的"
assert f"人均 {top['人均记录']:.0f} 条" in fig.texts[0].get_text(), \
    "标题里的人均不是从数据算出来的"
assert f"是 {ref['日期']} 的 {ratio:.0f} 倍" in fig.texts[0].get_text(), \
    "标题里的倍数不是从数据算出来的"

print("读回断言通过：三面板条形长度 = 中间表，标题数字来自数据")

### 构建数据质量核对表

每做完臆想检查，应该往dp_check_result表里记录一条，那个维度，查了什么,结果状态(通过，警告，不通过）证据是什么，怎么处理，D1-D5的质量报告就可以从这张表读取，不用回头翻notebook

In [ ]:
from signal_insight import quality

In [ ]:
top = d.loc[d['记录数'].idxmax()]
quality.dq_record(con,"DQ-01","时效性", "各日数据完整度", "ods_signal.timestamp",
        f"10月3日占{top['记录占比']:.1%}","警告",
        evidence="10-01仅54条，10-02 为1.4万条零散记录",
        decision="分析范围限定为 2018-10-03"
        )

### 10月3日每小时有多少数据

数据几乎都在10月3日这一天，这一天每个小时的记录数和用户数怎么变

In [ ]:
day3 = raw[raw['ts'].dt.strftime("%Y-%m-%d") == "2018-10-03"]
hourly = day3.groupby(day3["ts"].dt.hour).agg(
    记录数 = ("timestamp", "size"),
    用户数 = ('imsi', 'nunique')
).rename_axis("小时").reset_index()

assert hourly['记录数'].sum() == len(day3)
assert len(hourly) == 24

hourly.to_sql("ads_hourly_volume", con, if_exists="replace", index=False)

In [ ]:
from signal_insight.viz import *

In [ ]:
# ---------- 1. 指标：全部由代码计算 ----------
from matplotlib.ticker import FuncFormatter

h = hourly.set_index("小时").sort_index()
users, recs = h["用户数"], h["记录数"]
rpu = recs / users                                     # 人均记录数

pk_u_h, pk_u = int(users.idxmax()), int(users.max())   # 用户数峰值
lo_u_h, lo_u = int(users.idxmin()), int(users.min())   # 用户数谷值
pk_r_h, pk_r = int(recs.idxmax()), int(recs.max())     # 记录数峰值
u_drop = 1 - users[pk_r_h] / pk_u                      # 两个峰值之间用户减少
r_rise = pk_r / recs[pk_u_h] - 1                       # 两个峰值之间记录增加

N0, N1 = 0, 5                                          # 夜间时段 0–5 点
night_r = recs.loc[N0:N1]
night_share = night_r.sum() / recs.sum()               # 夜间记录占全天
night_lo_h = int(night_r.idxmin())                     # 夜间记录最低的小时
night_lo = int(night_r.min())

# ---------- 2. 标题、副标题、来源行 ----------
title = f"用户数{pk_u_h}点见顶（{pk_u:,}人），记录数到{pk_r_h}点才最高（{pk_r:,}条）"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = ("每小时的信令记录条数与去重用户数，各日合并；同一人在多个小时出现会分别计数，各小时用户数不能相加\n"
            "记录数取决于终端上报频率，不等于人数；两图纵轴均从 0 开始")
source = "数据来源：沈阳移动信令样本（脱敏，2018 年国庆）；制图: Azurekite 1210"

# ---------- 3. 画图：上下两个面板，共用横轴（不用双 Y 轴）----------
thousands = FuncFormatter(lambda v, _: f"{v:,.0f}")
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6.6), sharex=True)
fig.subplots_adjust(left=0.08, right=0.97, top=0.77, bottom=0.10, hspace=0.42)

for ax in (ax1, ax2):
    ax.axvspan(pk_u_h, pk_r_h, color=MAIN_LIGHT, lw=0, zorder=0)         # 浅蓝：人在减少、记录在增加
    ax.axvspan(N0, N1, color=GRAY, alpha=0.30, lw=0, zorder=0)           # 灰色：夜间 0–5 点
    ax.yaxis.set_major_formatter(thousands)
    ax.tick_params(labelsize=10)

# 上：用户数
ax1.plot(users.index, users.values, color=MAIN, lw=2.2)
ax1.scatter([pk_u_h], [pk_u], color=MAIN, s=36, zorder=3)
ax1.annotate(f"{pk_u_h} 点峰值 {pk_u:,} 人", (pk_u_h, pk_u), xytext=(0, 7),
             textcoords="offset points", ha="center", va="bottom",
             fontsize=10, color=MAIN, fontweight="bold")
ax1.scatter([lo_u_h], [lo_u], color=GRAY_DARK, s=24, zorder=3)
ax1.annotate(f"{lo_u_h} 点最低 {lo_u:,} 人", (lo_u_h, lo_u), xytext=(0, -8),
             textcoords="offset points", ha="center", va="top", fontsize=9.5, color=MUTED)
ax1.set_ylim(0, pk_u * 1.22)
ax1.text((N0 + N1) / 2, pk_u * 1.2, f"夜间 {N0}–{N1} 点", ha="center", va="top",
         fontsize=10, color=MUTED, fontweight="bold")
ax1.set_title("每小时用户数（人，去重）", loc="left", fontsize=10.5, color=INK, pad=6)

# 下：记录数，灰色打底，两个峰值之间那一段用主色强调
ax2.plot(recs.index, recs.values, color=GRAY_DARK, lw=2)
seg = recs.loc[pk_u_h:pk_r_h]
ax2.plot(seg.index, seg.values, color=MAIN, lw=2.6)
ax2.scatter([pk_r_h], [pk_r], color=MAIN, s=36, zorder=3)
ax2.annotate(f"{pk_r_h} 点峰值 {pk_r:,} 条", (pk_r_h, pk_r), xytext=(-8, 4),
             textcoords="offset points", ha="right", va="bottom",
             fontsize=10, color=MAIN, fontweight="bold")
ax2.text(pk_u_h - 0.4, pk_r * 1.02,
         f"{pk_u_h}→{pk_r_h} 点：用户减少 {u_drop:.0%}，记录增加 {r_rise:.0%}\n"
         f"人均记录从 {rpu[pk_u_h]:.1f} 条升到 {rpu[pk_r_h]:.1f} 条",
         ha="right", va="top", fontsize=9.5, color=INK, linespacing=1.5)
ax2.set_ylim(0, pk_r * 1.18)
ax2.text((N0 + N1) / 2, pk_r * 1.15,
         f"夜间 {N0}–{N1} 点\n仅占全天记录的 {night_share:.0%}\n{night_lo_h} 点降到最低 {night_lo:,} 条",
         ha="center", va="top", fontsize=9.5, color=MUTED, linespacing=1.5)
ax2.set_title("每小时信令记录数（条）", loc="left", fontsize=10.5, color=INK, pad=6)
ax2.set_xticks(range(0, 24, 3))
ax2.set_xticklabels([f"{x} 时" for x in range(0, 24, 3)])
ax2.set_xlim(-0.5, 23.6)

# ---------- 4. 强调色短条 + 标题 + 副标题 + 来源行 ----------
viz.finish(fig, title=title,
        subtitle=subtitle,
        source=source,
        path = "D01_03_每小时用户数与记录数.png"
          )
plt.show()

- 10月3日每小时信令量在凌晨5点降至最低(14,369条)，符合正常的作息。说明时间已经换正确换算为北京时间，凌晨仍有2400余名用户产生信令: 信令由网络自动触发，夜间同样可以定位，记录数在23点达到峰值(33,769条)。

| 编号 | 发现 | 证据 | 影响/决定 |
|---|---|---|---|
| F1 | 97.3%的记录都在10月3日，10月1日只有54条 | ads_daily_volume，图D1_01 | 分析范围订在10月3日 |
| F2 | 凌晨5点最低还有14369条信令 | ads_hourly_volume，图D1_02 | 信令有网络自动产生，夜里也能观察人在哪 → 判断过夜地在什么地方 |
| F3 | 23点记录最多（33769条），不符合一般作息 | 图D1_02 | 先查下数据是怎么取的，再下结论 |
| F4 | tmp0、tmp1两列没有任何内容 | 数据画像 | 后续过程直接删除 |

In [ ]:
con.close() # 用完关闭数据库连接